Installation

In [ ]:
!pip install "pandas<3.0.0" "google-genai" -U

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Loading GEMINI API KEY

In [ ]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)

###**Bio Generation**

get speaker bio

In [ ]:
def get_speaker_bio(speaker_name, scene_text):
  prompt = f"""Given this conversation between speakers:
{scene_text}
In overall above conversation, what do you think about the characteristics of speaker {speaker_name} Note: provide an answer within 250 words"""
  response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
  return response.text.strip()

In [ ]:
def predict_emotion(bio, context, utterance):
    # Alphabetical labels as requested
    labels = "Anger, Disgust, Fear, Joy, Neutral, Sadness, Surprise"

    prompt = f"""
    You are an expert in emotion recognition.

    SPEAKER PERSONALITY:
    {bio}

    CONVERSATION CONTEXT:
    {context}

    TARGET LINE:
    "{utterance}"

    Based on the speaker's personality and the context, which emotion is expressed?
    Choose ONLY from: {labels}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text.strip()

processing data

In [ ]:
import pandas as pd
import os
import time
from tqdm.auto import tqdm

def process_meld(file_path, output_csv="checkpoint_results.csv"):

    df = pd.read_csv(file_path)
    df = df.sort_values(['Dialogue_ID', 'Utterance_ID'])

    results = []
    processed_ids = set()

    # RESUME LOGIC: Check if we already have some work done
    if os.path.exists(output_csv):
        results_df = pd.read_csv(output_csv)
        results = results_df.to_dict('records')
        processed_ids = set(results_df['Dialogue_ID'].unique())
        print(f"Resuming... {len(processed_ids)} dialogues already finished.")

    grouped = df.groupby(['Dialogue_ID'])

    for diag_id, group in tqdm(grouped, desc="Processing Scenes"):
        # SKIP dialogues we already processed
        if diag_id in processed_ids:
            continue

        full_scene_text = "\n".join([f"{row['Speaker']}: {row['Utterance']}" for _, row in group.iterrows()])

        try:
            # 1. Generate bios once per speaker
            speakers = group['Speaker'].unique()
            speaker_bios_for_this_scene = {}
            for speaker in speakers:
                speaker_bios_for_this_scene[speaker] = get_speaker_bio(speaker, full_scene_text)
                print(f"Bio for {speaker}:\n{speaker_bios_for_this_scene[speaker]}")
                time.sleep(0.1) # 6 seconds = 10 requests per minute (very safe)

            # 2. Predict emotion for each line
            group_list = group.to_dict('records')
            for i, row in enumerate(group_list):
                start = max(0, i - 3)
                context_snippet = "\n".join([f"{r['Speaker']}: {r['Utterance']}" for r in group_list[start:i]])
                current_bio = speaker_bios_for_this_scene[row['Speaker']]

                prediction = predict_emotion(current_bio, context_snippet, row['Utterance'])
                time.sleep(0.1) # Pacing

                results.append({
                    'Dialogue_ID': diag_id,
                    'Utterance_ID': row['Utterance_ID'],
                    'Speaker': row['Speaker'],
                    'Original_Emotion': row['Emotion'],
                    'Predicted_Emotion': prediction
                })

            # Save after EVERY dialogue so you never lose progress
            pd.DataFrame(results).to_csv(output_csv, index=False)

        except Exception as e:
            # If we hit the 429 error again, it will save and stop gracefully
            print(f"\nQuota hit or Error at Dialogue {diag_id}: {e}")
            pd.DataFrame(results).to_csv(output_csv, index=False)
            return pd.DataFrame(results)

    return pd.DataFrame(results)

test_df = pd.read_csv('/content/drive/MyDrive/test_sent_emo.csv')

# Get the first 50 unique dialogues
sample_df = test_df[test_df['Dialogue_ID'] < 50] # First 50 dialogues
sample_df.to_csv('sample_input.csv', index=False)

# 3. Call the function
final_results_df = process_meld('sample_input.csv', output_csv='sample.csv')

# 4. Success message
print(f"Done! Results saved to {'sample.csv'}")
final_results_df.head()

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
import re

def normalize_predicted_emotion(predicted_text, valid_labels):
    predicted_text_lower = predicted_text.lower()
    for label in valid_labels:
        if label in predicted_text_lower:
            return label
    # Try to extract words and see if they match a label (e.g., for '**Joy**')
    words = re.findall(r'\b\w+\b', predicted_text_lower)
    for word in words:
        if word in valid_labels:
            return word
    return 'neutral' # Default to neutral if no clear emotion is found

# Get the list of unique original emotions to use as valid labels
valid_original_emotions = final_results_df['Original_Emotion'].unique()

# Apply normalization to the 'Predicted_Emotion' column
final_results_df['Predicted_Emotion_Normalized'] = final_results_df['Predicted_Emotion'].apply(
    lambda x: normalize_predicted_emotion(x, valid_original_emotions)
)

# Recalculate accuracy using the normalized predictions
accuracy = accuracy_score(final_results_df['Original_Emotion'], final_results_df['Predicted_Emotion_Normalized'])
print(f"Accuracy: {accuracy:.4f}")

# Recalculate precision, recall, and F1-score for each class
precision, recall, f1_score, _ = precision_recall_fscore_support(
    final_results_df['Original_Emotion'],
    final_results_df['Predicted_Emotion_Normalized'],
    average='weighted',
    zero_division=0
)

print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted Recall: {recall:.4f}")
print(f"Weighted F1-Score: {f1_score:.4f}")

per_class_precision, per_class_recall, per_class_f1_score, per_class_support = precision_recall_fscore_support(
    final_results_df['Original_Emotion'],
    final_results_df['Predicted_Emotion_Normalized'],
    zero_division=0
)

labels = sorted(final_results_df['Original_Emotion'].unique())
metrics_df = pd.DataFrame({
    'Emotion': labels,
    'Precision': per_class_precision,
    'Recall': per_class_recall,
    'F1-Score': per_class_f1_score,
    'Support': per_class_support
})

print("\nPer-Class Metrics:")
display(metrics_df)


STOP HERE?


In [ ]:
import os

# Delete the empty files so they can be recreated correctly
for file in ['sample_input.csv', 'sample.csv']:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted empty file: {file}")

In [ ]:
print('Re-running BiosERC replication with updated prompts...')
# Ensure clean slate for re-run by deleting existing output file
if os.path.exists('sample.csv'):
    os.remove('sample.csv')

final_results_df = process_meld('sample_input.csv', output_csv='sample.csv')
print(f"Done! Results saved to {'sample.csv'}")
final_results_df.head()
